# Fine-tuning de Qwen 2.5 3B para clasificación de relevancia RAG

## Objetivo del notebook

Este notebook implementa el **fine-tuning con QLoRA** del modelo Qwen 2.5 3B-Instruct para la tarea de
**clasificación de relevancia de documentos** en el pipeline RAG de NormaBot.

El modelo aprenderá a decidir, dado un par `(consulta, documento)`, si el documento contiene
información útil para responder la consulta. La salida es binaria:
- **`relevante`** → el documento debe incluirse en el contexto de generación
- **`no relevante`** → el documento debe descartarse

## Contexto en NormaBot

El pipeline RAG (`src/rag/main.py`) sigue el flujo:

```
Retrieve (ChromaDB) → Grade (Qwen 2.5 3B via Ollama) → Generate (Bedrock Nova Lite)
```

La etapa de **grading** actual usa el modelo base con prompting directo. Este fine-tuning busca
especializarlo en el dominio legal (EU AI Act, BOE) para reducir falsos positivos en el contexto
de generación. El experimento:

1. Establece un **baseline** con el modelo base + prompting
2. **Fine-tunea** el mismo modelo con QLoRA sobre el dataset de relevancia
3. **Compara** ambos enfoques: Accuracy, Precision, Recall, F1-macro
4. Proporciona el **código de integración** listo para `src/rag/main.py`

> Notebook monolítico (referencia). Los notebooks individuales `01–04` son la versión modular activa.

# Elección del modelo: Qwen 2.5 3B-Instruct

**¿Por qué Qwen 2.5 3B?**
- Ya está en producción en NormaBot via Ollama para la etapa de grading
- Open-weight, totalmente compatible con LoRA y QLoRA
- Suficientemente ligero para entrenarlo en una GPU con ≥ 8 GB VRAM
- Muy buena capacidad semántica en español para su tamaño
- La tarea es clasificación binaria simple, no requiere un modelo más grande

**Parámetros:** ~3.000 millones  
**Cuantización durante entrenamiento:** 4-bit NF4 (QLoRA, `bitsandbytes`)  
**Adaptadores LoRA:** r=8, alpha=16, capas de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`)

> Entrenado localmente en **NVIDIA RTX 4070 Super (13 GB VRAM)**.  
> Compatible con cualquier GPU ≥ 8 GB VRAM (RTX 3080, A10G, etc.).

# Instalación de librerías y comprobación de entorno


In [ ]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path

print(f"PyTorch:          {torch.__version__}")
print(f"CUDA disponible:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM total:       {total_vram:.1f} GB")
else:
    print("Sin GPU — el entrenamiento será extremadamente lento. Se recomienda GPU con >= 8 GB VRAM.")

# Dataset de relevancia

El dataset está generado en `data/processed/grading_dataset.jsonl` mediante
`data/generate_grading_dataset.py`. Cada línea es un ejemplo con tres campos:

```json
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Artículo 5...", "label": "relevante"}
{"query": "¿Qué sistemas de IA están prohibidos?", "document": "Real Decreto 463/2020...", "label": "no relevante"}
```

| Campo | Tipo | Descripción |
|-------|------|-------------|
| `query` | str | Consulta del usuario sobre normativa de IA |
| `document` | str | Fragmento de texto recuperado de ChromaDB |
| `label` | str | `"relevante"` o `"no relevante"` |

**Composición:** 117 queries · 634 ejemplos · 44.6% relevantes / 55.4% no relevantes  
**Fuentes:** EU AI Act (Arts. 5, 6, 9–17, 25–26, 43, 47–53, 72–73, 99, 113), AESIA, RGPD/LOPDGDD  
**Negativos:** hard negatives del dominio (otro artículo legal) + easy negatives (otra ley ajena al dominio)


In [ ]:
import os
from pathlib import Path

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:

# Verificar que el dataset existe antes de continuar
assert DATASET_PATH.exists(), (
    f"Dataset no encontrado en {DATASET_PATH.resolve()}\n"
    "Genera el dataset ejecutando desde la raíz del repo:\n"
    "  python data/generate_grading_dataset.py"
)

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f if line.strip()]

n_rel = sum(1 for ex in all_data if ex["label"] == LABEL_RELEVANTE)
n_no  = len(all_data) - n_rel

print(f"Dataset cargado: {DATASET_PATH.resolve()}")
print(f"  Total ejemplos:  {len(all_data)}")
print(f"  Relevantes:      {n_rel} ({n_rel/len(all_data)*100:.1f}%)")
print(f"  No relevantes:   {n_no} ({n_no/len(all_data)*100:.1f}%)")
print()
print("Muestra:")
print(json.dumps(all_data[0], ensure_ascii=False, indent=2))


In [ ]:
from sklearn.model_selection import train_test_split

# División estratificada train / val / test (70 / 15 / 15)
labels_all = [ex["label"] for ex in all_data]

train_data, temp = train_test_split(
    all_data, test_size=0.30, random_state=42, stratify=labels_all
)
labels_temp = [ex["label"] for ex in temp]
val_data, test_data = train_test_split(
    temp, test_size=0.50, random_state=42, stratify=labels_temp
)

print("Split del dataset:")
print(f"  Train: {len(train_data)} ejemplos")
print(f"  Val:   {len(val_data)} ejemplos")
print(f"  Test:  {len(test_data)} ejemplos")


In [ ]:
def show_split_stats(data: list[dict], name: str):
    df = pd.DataFrame(data)
    print(f"\n{'='*55}")
    print(f"  {name} ({len(df)} ejemplos)")
    print(f"{'='*55}")
    print(df["label"].value_counts().to_string())
    print(f"  Longitud media query:    {df['query'].str.len().mean():.0f} chars")
    print(f"  Longitud media document: {df['document'].str.len().mean():.0f} chars")


show_split_stats(train_data, "TRAIN")
show_split_stats(val_data,   "VALIDATION")
show_split_stats(test_data,  "TEST")

print(f"\n{'='*55}")
print("Ejemplo del dataset:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

# Evaluación baseline: modelo base sin fine-tunear

Antes de entrenar, evaluamos el rendimiento del modelo base con **prompting directo**.
Esto establece el punto de comparación para medir el impacto del fine-tuning.

El prompt de grading que usamos aquí es equivalente al que ya usa `src/rag/main.py`,
pero adaptado para producir `"relevante"` / `"no relevante"` en lugar de `"si"` / `"no"`.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Prompt del sistema — equivalente al GRADING_PROMPT actual en src/rag/main.py
GRADING_SYSTEM_PROMPT = (
    "Eres un asistente especializado en normativa de inteligencia artificial. "
    "Tu tarea es determinar si un documento contiene información útil para responder "
    "una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). "
    "Responde únicamente con 'relevante' o 'no relevante', sin explicación adicional."
)


def build_grading_messages(query: str, document: str) -> list[dict]:
    """Construye los mensajes en formato chat para el modelo de grading."""
    return [
        {"role": "system", "content": GRADING_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Consulta: {query}\n\n"
                f"Documento: {document}\n\n"
                "¿Es este documento relevante para responder la consulta?"
            ),
        },
    ]


print("Cargando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizador listo. Vocabulario: {tokenizer.vocab_size:,} tokens")

# Carga del modelo base en 4-bit para la evaluación baseline
bnb_eval_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"\nCargando modelo base {MODEL_ID} (4-bit) para evaluación baseline...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_eval_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()
print("✓ Modelo base cargado.")

In [ ]:
def predict_relevance(query: str, document: str, model, tok,
                      max_new_tokens: int = 12) -> str:
    """
    Predice si un documento es relevante para una consulta.
    Devuelve LABEL_RELEVANTE o LABEL_NO_RELEVANTE.
    Funciona tanto con el modelo base como con el fine-tuneado.
    """
    messages = build_grading_messages(query, document)
    text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tok.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Normalizar: si contiene "no relevante" → no relevante; si solo "relevante" → relevante
    if "no relevante" in response or "no es relevante" in response:
        return LABEL_NO_RELEVANTE
    elif "relevante" in response:
        return LABEL_RELEVANTE
    else:
        # Respuesta inesperada: más conservador → no relevante
        return LABEL_NO_RELEVANTE


# Prueba rápida
sample = test_data[0]
pred = predict_relevance(sample["query"], sample["document"], base_model, tokenizer)
print(f"Predicción baseline (muestra):")
print(f"  Query:    {sample['query'][:80]}...")
print(f"  Predicho: {pred}")
print(f"  Real:     {sample['label']}")

In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score

print(f"Evaluando baseline en {len(test_data)} ejemplos del test set...")
print("(Puede tardar varios minutos)")

baseline_preds = []
baseline_true  = []

for i, example in enumerate(test_data):
    pred = predict_relevance(
        example["query"], example["document"], base_model, tokenizer
    )
    baseline_preds.append(pred)
    baseline_true.append(example["label"])
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{len(test_data)} completados")

baseline_acc = accuracy_score(baseline_true, baseline_preds)
baseline_f1  = f1_score(baseline_true, baseline_preds, average="macro", zero_division=0)

print(f"\n{'='*55}")
print("BASELINE — Qwen 2.5 3B-Instruct (sin fine-tuning)")
print(f"{'='*55}")
print(f"Accuracy:  {baseline_acc:.4f}")
print(f"F1-macro:  {baseline_f1:.4f}")
print()
print(classification_report(baseline_true, baseline_preds, labels=LABELS, zero_division=0))

# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [ ]:
def format_training_prompt(example: dict) -> str:
    """
    Convierte un ejemplo del dataset en el texto completo de entrenamiento,
    incluyendo la respuesta del assistant y el token de fin de secuencia.
    """
    messages = build_grading_messages(example["query"], example["document"])
    # Aplicar el chat template hasta el turno del assistant
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Añadir la respuesta esperada del assistant
    text += example["label"] + tokenizer.eos_token
    return text


# Verificar un ejemplo
sample_formatted = format_training_prompt(train_data[0])
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

In [ ]:
from datasets import Dataset


def examples_to_hf_dataset(examples: list[dict]) -> Dataset:
    """Convierte la lista de ejemplos a un Dataset de HuggingFace con columna 'text'."""
    records = [{"text": format_training_prompt(ex)} for ex in examples]
    return Dataset.from_list(records)


train_dataset = examples_to_hf_dataset(train_data)
val_dataset   = examples_to_hf_dataset(val_data)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validación:    {len(val_dataset)} ejemplos")
print()
print("Primeros 300 chars del primer ejemplo de train:")
print(train_dataset[0]["text"][:300])

# Fine-tuning con QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) combina:
- **Cuantización 4-bit NF4** para reducir el uso de VRAM
- **LoRA** para entrenar solo un subconjunto pequeño de parámetros

Con r=8 entrenamos aproximadamente **el 1-2% de los parámetros totales** del modelo,
lo que hace el fine-tuning viable en una GPU T4 (16 GB VRAM).

Usamos `SFTTrainer` de `trl` (Supervised Fine-Tuning Trainer), que gestiona automáticamente
el packing de secuencias, la máscara de padding y la pérdida solo sobre el turno de assistant.


In [ ]:
# Liberar el modelo baseline de memoria antes de cargar el modelo de entrenamiento
try:
    del base_model
    torch.cuda.empty_cache()
    print("✓ Modelo baseline liberado de VRAM.")
except NameError:
    pass

# Configuración de cuantización 4-bit para QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                  # NormalFloat4, mejor para pesos normalmente distribuidos
    bnb_4bit_compute_dtype=torch.float16,        # computación en fp16 para velocidad
    bnb_4bit_use_double_quant=True,              # doble cuantización para ahorrar ~0.4 bits/parámetro extra
)

print(f"Cargando {MODEL_ID} con QLoRA (4-bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False          # desactivar KV-cache durante entrenamiento
model.config.pretraining_tp = 1         # tensor parallelism = 1 (single GPU)

print(f"✓ Modelo cargado. Dispositivo: {model.device}")

In [ ]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# Preparar modelo para entrenamiento en k-bit (activa gradient checkpointing)
model = prepare_model_for_kbit_training(model)

# Configuración LoRA
# - r=8: rank de los adaptadores. Mayor r → más parámetros entrenables, más capacidad
# - lora_alpha=16: escala de los pesos LoRA. Convención: alpha = 2 * r
# - target_modules: capas de atención y feedforward de Qwen 2.5
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # atención multi-cabeza
        "gate_proj", "up_proj", "down_proj",         # feedforward (SwiGLU)
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig

# SFTConfig reemplaza TrainingArguments en trl >= 0.12:
# dataset_text_field, max_seq_length y packing se definen aquí, no en SFTTrainer
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
)

print("SFTConfig configurado:")
print(f"  Epochs:         {training_args.num_train_epochs}")
print(f"  Batch efectivo: {training_args.per_device_train_batch_size} × {training_args.gradient_accumulation_steps} = {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate:  {training_args.learning_rate}")
print(f"  Max seq len:    {training_args.max_seq_length}")
print(f"  Output dir:     {OUTPUT_DIR}")

In [ ]:
from trl import SFTTrainer

# En trl >= 0.12: processing_class reemplaza tokenizer,
# y dataset_text_field/max_seq_length/packing van en SFTConfig (training_args)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Iniciando entrenamiento QLoRA...")
steps = (
    len(train_dataset)
    // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
    * training_args.num_train_epochs
)
print(f"  Pasos estimados: {steps}")

train_result = trainer.train()

print(f"\nEntrenamiento completado.")
print(f"  Loss final:    {train_result.training_loss:.4f}")
print(f"  Tiempo total:  {train_result.metrics.get('train_runtime', 0):.1f}s")

# Evaluación del modelo fine-tuneado

Evaluamos el modelo con los mismos ejemplos de test que usamos para el baseline,
para poder comparar directamente ambos enfoques.


In [ ]:
model.eval()

print(f"Evaluando modelo fine-tuneado en {len(test_data)} ejemplos...")

finetuned_preds = []
finetuned_true  = []

for i, example in enumerate(test_data):
    pred = predict_relevance(
        example["query"], example["document"], model, tokenizer
    )
    finetuned_preds.append(pred)
    finetuned_true.append(example["label"])
    if (i + 1) % 5 == 0:
        print(f"  {i + 1}/{len(test_data)} completados")

finetuned_acc = accuracy_score(finetuned_true, finetuned_preds)
finetuned_f1  = f1_score(finetuned_true, finetuned_preds, average="macro", zero_division=0)

print(f"\n{'='*55}")
print("FINE-TUNED — Qwen 2.5 3B + QLoRA")
print(f"{'='*55}")
print(f"Accuracy:  {finetuned_acc:.4f}")
print(f"F1-macro:  {finetuned_f1:.4f}")
print()
print(classification_report(finetuned_true, finetuned_preds, labels=LABELS, zero_division=0))

In [ ]:
# Tabla comparativa baseline vs fine-tuned
mejora_acc = finetuned_acc - baseline_acc
mejora_f1  = finetuned_f1  - baseline_f1

comparison = pd.DataFrame({
    "Modelo": [
        "Qwen 2.5 3B (baseline, prompting)",
        "Qwen 2.5 3B + QLoRA (fine-tuned)",
        "Mejora",
    ],
    "Accuracy": [baseline_acc, finetuned_acc, mejora_acc],
    "F1-macro": [baseline_f1, finetuned_f1, mejora_f1],
}).set_index("Modelo").round(4)

print("\nComparativa de modelos:")
print(comparison.to_string())

if mejora_f1 > 0:
    print(f"\n✓ El fine-tuning mejora el F1-macro en {mejora_f1:+.4f} ({mejora_f1 / (baseline_f1 + 1e-9) * 100:.1f}% relativo).")
else:
    print(f"\n⚠️  El fine-tuning no mejoró el baseline (Δ F1-macro = {mejora_f1:+.4f}).")
    print("   Posibles causas: dataset muy pequeño, hiperparámetros, prompt no óptimo.")

# Seguimiento con MLflow

Registramos los parámetros y métricas en el servidor MLflow del proyecto
(configurado via `MLFLOW_TRACKING_URI` en `.env`), siguiendo el mismo patrón
que el clasificador de riesgo (`src/classifier/`).


In [ ]:
import mlflow
from dotenv import load_dotenv

# En Colab: define MLFLOW_TRACKING_URI como variable de entorno o en Secrets
load_dotenv()
MLFLOW_EXPERIMENT_NAME = "rag_grader_relevancia"

try:
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run(run_name="qwen25-3b-qlora-grader") as run:
        mlflow.log_params({
            "model_id":            MODEL_ID,
            "lora_r":              lora_config.r,
            "lora_alpha":          lora_config.lora_alpha,
            "lora_dropout":        lora_config.lora_dropout,
            "target_modules":      ",".join(lora_config.target_modules),
            "epochs":              training_args.num_train_epochs,
            "learning_rate":       training_args.learning_rate,
            "batch_size_efectivo": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
            "max_seq_len":         MAX_SEQ_LEN,
            "train_size":          len(train_data),
            "val_size":            len(val_data),
            "test_size":           len(test_data),
        })
        mlflow.log_metrics({
            "baseline_accuracy":  baseline_acc,
            "baseline_f1_macro":  baseline_f1,
            "finetuned_accuracy": finetuned_acc,
            "finetuned_f1_macro": finetuned_f1,
            "mejora_f1_macro":    mejora_f1,
            "train_loss":         train_result.training_loss,
        })

        print(f"Metricas registradas en MLflow.")
        print(f"  Experimento: {MLFLOW_EXPERIMENT_NAME}")
        print(f"  Run ID:      {run.info.run_id}")
        print(f"  URI:         {tracking_uri}")

except Exception as e:
    print(f"MLflow no disponible: {e}")
    print(f"  Accuracy: {finetuned_acc:.4f} | F1-macro: {finetuned_f1:.4f}")

# Guardado del modelo

Guardamos el **adaptador LoRA** en `src/finetuning/output/qwen-grader-lora/adapter_final/`.
El adaptador pesa solo unos pocos MB y es suficiente para inferencia si se carga junto al modelo base.

Para producción con Ollama, descomenta el bloque de merge para obtener el modelo completo
y convertirlo a GGUF.

In [ ]:
import os
os.makedirs(ADAPTER_PATH, exist_ok=True)

# Guardar adaptador LoRA en disco
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adaptador LoRA guardado en: {ADAPTER_PATH}")

# Guardar metadatos del modelo
metadata = {
    "model_id":              MODEL_ID,
    "task":                  "rag_relevance_grading",
    "labels":                LABELS,
    "lora_r":                lora_config.r,
    "lora_alpha":            lora_config.lora_alpha,
    "dataset_path":          str(DATASET_PATH),
    "train_size":            len(train_data),
    "val_size":              len(val_data),
    "test_size":             len(test_data),
    "test_accuracy":         round(finetuned_acc, 4),
    "test_f1_macro":         round(finetuned_f1, 4),
    "baseline_f1_macro":     round(baseline_f1, 4),
    "mejora_f1_macro":       round(mejora_f1, 4),
    "grading_system_prompt": GRADING_SYSTEM_PROMPT,
}

with open(f"{ADAPTER_PATH}/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"Metadatos guardados en: {ADAPTER_PATH}/model_metadata.json")

# ============================================================
# OPCIONAL: Merge del adaptador con el modelo base
# Necesario si se quiere convertir a GGUF y servir via Ollama.
# ============================================================
# os.makedirs(MERGED_PATH, exist_ok=True)
# from peft import PeftModel
# print("Merging adaptador con modelo base (requiere ~7 GB de RAM)...")
# base_for_merge = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, torch_dtype=torch.float16, device_map="auto"
# )
# merged = PeftModel.from_pretrained(base_for_merge, ADAPTER_PATH)
# merged = merged.merge_and_unload()
# merged.save_pretrained(MERGED_PATH)
# tokenizer.save_pretrained(MERGED_PATH)
# print(f"Modelo merged guardado en: {MERGED_PATH}")
# print("  Siguiente paso: convertir a GGUF con llama.cpp y subir a Ollama.")

# Integración con el orquestador (`src/rag/main.py`)

El modelo fine-tuneado reemplaza al modelo base en la función `grade()` de `src/rag/main.py`.
La salida cambia de `"si"`/`"no"` a `"relevante"`/`"no relevante"` para mayor claridad.

## Opciones de integración

| Opción | Cuándo usarla |
|--------|---------------|
| **A — Carga directa PEFT** | Desarrollo, testing, entornos con GPU propia |
| **B — Convertir a GGUF + Ollama** | Producción (mismo setup que el modelo actual) |

El fragmento de código de abajo muestra la **Opción A**.
Para la Opción B, hacer merge del adaptador (celda anterior), convertir con `llama.cpp`,
y actualizar el `model` en `_get_grading_llm()` a `qwen2.5-grader:3b`.


In [ ]:
# ============================================================
# Fragmento de integración para src/rag/main.py
# ============================================================
# Copiar este código en src/rag/main.py para activar el grader fine-tuneado.
# Requiere que el adaptador esté en FINETUNED_ADAPTER_PATH.
# ============================================================

INTEGRATION_SNIPPET = '''
# ---- Añadir en src/rag/main.py ----

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

FINETUNED_ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"

GRADING_SYSTEM_PROMPT_FT = (
    "Eres un asistente especializado en normativa de inteligencia artificial. "
    "Tu tarea es determinar si un documento contiene información útil para responder "
    "una consulta sobre regulación de IA (EU AI Act, BOE, normativa española). "
    "Responde únicamente con \'relevante\' o \'no relevante\', sin explicación adicional."
)

_grading_model_ft = None
_grading_tok_ft   = None


def _get_grading_model_finetuned():
    """Carga el grader fine-tuneado (Qwen 2.5 3B + LoRA) — singleton."""
    global _grading_model_ft, _grading_tok_ft
    if _grading_model_ft is None:
        base = AutoModelForCausalLM.from_pretrained(
            "Qwen/Qwen2.5-3B-Instruct",
            device_map="auto",
            torch_dtype=torch.float16,
        )
        _grading_model_ft = PeftModel.from_pretrained(base, FINETUNED_ADAPTER_PATH)
        _grading_model_ft.eval()
        _grading_tok_ft = AutoTokenizer.from_pretrained(FINETUNED_ADAPTER_PATH)
    return _grading_model_ft, _grading_tok_ft


def grade_with_finetuned(query: str, docs: list[dict], threshold: float = 0.7) -> list[dict]:
    """
    Versión actualizada de grade() que usa el modelo fine-tuneado.
    Salida: "relevante" / "no relevante" (en lugar de "si" / "no").
    Fallback a filtro por score si el modelo no está disponible.
    """
    if not docs:
        return []

    try:
        model_ft, tok_ft = _get_grading_model_finetuned()
    except Exception:
        logger.warning("Grader fine-tuneado no disponible, usando fallback por score")
        return _grade_by_score(docs, threshold)

    relevant = []
    for doc in docs:
        messages = [
            {"role": "system", "content": GRADING_SYSTEM_PROMPT_FT},
            {"role": "user", "content": (
                f"Consulta: {query}\\n\\n"
                f"Documento: {doc[\'doc\']}\\n\\n"
                "¿Es este documento relevante para responder la consulta?"
            )},
        ]
        text   = tok_ft.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok_ft(text, return_tensors="pt").to(model_ft.device)

        with torch.no_grad():
            out = model_ft.generate(**inputs, max_new_tokens=12, do_sample=False,
                                    pad_token_id=tok_ft.eos_token_id)
        gen      = out[0][inputs["input_ids"].shape[1]:]
        response = tok_ft.decode(gen, skip_special_tokens=True).strip().lower()

        if "no relevante" not in response and "relevante" in response:
            relevant.append(doc)

    return relevant
'''

print(INTEGRATION_SNIPPET)

In [ ]:
# Verificación: recargar el adaptador desde disco y hacer una predicción de prueba
from peft import PeftModel

print(f"Verificando adaptador en: {ADAPTER_PATH}")

try:
    del model
    torch.cuda.empty_cache()
except NameError:
    pass

base_for_test = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)
loaded_model = PeftModel.from_pretrained(base_for_test, ADAPTER_PATH)
loaded_model.eval()
loaded_tok = AutoTokenizer.from_pretrained(ADAPTER_PATH)

print("\nPruebas de inferencia con modelo recargado desde disco:")
for ex in [test_data[0], test_data[-1]]:
    pred = predict_relevance(ex["query"], ex["document"], loaded_model, loaded_tok)
    ok = pred == ex["label"]
    print(f"  {'OK' if ok else 'FAIL'}  Real: {ex['label']:<15} | Predicho: {pred}")

print("\nAdaptador cargado y funcionando correctamente.")

# Conclusiones

## Resultados

| Modelo | Accuracy | F1-macro |
|--------|:--------:|:--------:|
| Qwen 2.5 3B-Instruct (baseline, prompting) | 0.8542 | 0.8500 |
| Qwen 2.5 3B-Instruct + QLoRA (fine-tuned) | **0.9062** | **0.9057** |
| Mejora | +0.0521 | **+0.0557** |

### Por clase

| Modelo | Clase | Precision | Recall | F1 |
|--------|-------|:---------:|:------:|:--:|
| Baseline | relevante | 0.89 | 0.77 | 0.82 |
| Baseline | no relevante | 0.83 | 0.92 | 0.88 |
| Fine-tuned | relevante | 0.87 | **0.93** | **0.90** |
| Fine-tuned | no relevante | **0.94** | 0.89 | **0.91** |

El fine-tuning mejora sobre todo el **recall de "relevante"** (+0.16) y la **precision de "no relevante"** (+0.11).

### Entrenamiento

- **Modelo base:** Qwen 2.5 3B-Instruct | Cuantización: 4-bit NF4 (QLoRA)
- **Parámetros entrenables:** 14,966,784 (0.48% de 3.1B)
- **Loss final:** 0.7306 | **Tiempo:** 1435 s (~24 min en RTX 4070 Super)
- **Épocas:** 3 | **Batch efectivo:** 16 | **LR:** 2e-4 (cosine)

## Dataset utilizado

- **634 ejemplos** · 117 queries sobre EU AI Act, RGPD y AESIA
- Split: 70% train (443) / 15% val (95) / 15% test (96)
- Balanceo: 44.6% relevantes / 55.4% no relevantes
- Negativos: hard negatives del mismo dominio legal + easy negatives de leyes ajenas
- Generado con `data/generate_grading_dataset.py`

## Limitaciones

- **Tamaño del dataset**: 634 ejemplos es suficiente para un primer experimento pero modesto.
  Ampliar con más queries o con datos reales del retriever ChromaDB mejorará la generalización.
- **Datos sintéticos**: los fragmentos documentales del dataset se generaron a partir de las fuentes
  reales (EU AI Act, RGPD, AESIA) pero no son los chunks exactos de ChromaDB.
  Idealmente el dataset final usaría los chunks reales que devuelve el retriever en producción.
- **Integración con Ollama**: el adaptador LoRA requiere cargar el modelo base en memoria.
  Para producción, hacer merge y convertir a GGUF.

## Próximos pasos

1. **Integrar** en `src/rag/main.py` con la función `grade_with_finetuned()` (fragmento en celda `integration-code-31`)
2. **Ampliar el dataset** con chunks reales del retriever ChromaDB
3. *(Opcional)* Merge + conversión a GGUF + push a Ollama para producción

## Impacto en el pipeline

Un grader especializado reduce el ruido en el contexto de generación:
- Menos documentos irrelevantes → respuestas más precisas y menos alucinaciones en citas legales
- Menor coste de tokens en la llamada a Bedrock Nova Lite (contexto más corto)

---
_Informe preliminar generado por IA. Consulte profesional jurídico._